In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as st

%matplotlib inline
Web_data_1 = pd.read_csv('../data/raw/df_final_web_data_pt_1.txt')
Web_data_2 = pd.read_csv('../data/raw/df_final_web_data_pt_2.txt')
Final_demo = pd.read_csv('../data/raw/df_final_demo.txt')
Clients = pd.read_csv('../data/raw/df_final_experiment_clients.txt')

In [ ]:
def combining (data1,data2,data3,data4):

In [2]:
combined_df = pd.concat([Web_data_1, Web_data_2], ignore_index=True)
merged = combined_df.merge(Clients, on='client_id', how='inner')
df = merged.merge(Final_demo, on='client_id', how='inner')
df.to_csv("../data/clean/data_merged",index =False)

In [34]:
def engagement_level(row):
    score = (
        row['logons_6_mnth'] +
        row['calls_6_mnth'] +
        row['num_accts']
    )
    if score >= 14:
        return 'High'
    elif score >= 10:
        return 'Medium'
    else:
        return 'Low'

In [24]:
df['engagement_level'] = df.apply(engagement_level, axis=1)

In [8]:
df_sorted = df.sort_values(by='date_time', ascending=False)
df_unique = df_sorted.drop_duplicates(subset='client_id', keep='first')

In [25]:
df_unique.head()

,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,engagement_level
385752,6187864,113539532_90731779729,77393632_6804608909_354572,step_2,2017-06-20 23:57:06,Control,13.0,163.0,55.0,F,2.0,174412.75,4.0,7.0,High
445470,8295888,238902635_81860264789,540045810_7898209765_29471,confirm,2017-06-20 23:44:20,NaN,6.0,81.0,48.0,U,3.0,89987.63,7.0,7.0,High
421492,2142847,394575357_33279393282,376590287_73561231479_860065,confirm,2017-06-20 23:40:31,NaN,4.0,57.0,42.0,F,2.0,29615.15,5.0,5.0,High
418424,6868690,790824480_59095058992,220987639_83305611081_845764,start,2017-06-20 23:32:41,NaN,12.0,154.0,27.0,M,2.0,78598.77,3.0,3.0,Low
423101,6506786,937099049_82984578854,603535171_56728735321_558098,confirm,2017-06-20 23:30:14,NaN,23.0,286.0,29.0,M,3.0,402521.26,7.0,7.0,High


In [21]:
engagement_stat = df_sorted.agg({
    "num_accts": ["mean", "median"],
    "calls_6_mnth": ["mean", "median"],
    "logons_6_mnth": ["mean","median"]})
engagement_stat

,num_accts,calls_6_mnth,logons_6_mnth
mean,2.264767,3.529176,5.709877
median,2.000000,3.000000,6.000000


In [28]:
counts = df_unique['engagement_level'].value_counts()
counts

engagement_level
Low       33407
High      32043
Medium     5159
Name: count, dtype: int64

In [53]:
df_unique.to_csv("data_engagement.csv",index =False)

In [32]:
df_unique[['clnt_age', 'clnt_tenure_yr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']].mean()

clnt_age              46.442240
clnt_tenure_yr        12.052950
num_accts              2.255528
bal               147445.240641
calls_6_mnth           3.382478
logons_6_mnth          5.566740
dtype: float64

In [33]:
df.groupby('gendr')[['clnt_age', 'clnt_tenure_yr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']].mean().round(1)

,clnt_age,clnt_tenure_yr,num_accts,bal,calls_6_mnth,logons_6_mnth
gendr,,,,,,
F,50.7,15.0,2.2,143613.2,3.2,5.4
M,49.8,15.1,2.3,234590.3,3.9,6.0
U,42.6,6.7,2.2,100422.4,3.5,5.6
X,42.1,9.7,2.4,27685.1,4.1,6.2


In [51]:
def engagement_score(row):
    return (
        row['logons_6_mnth'] +
        row['calls_6_mnth'] +
        row['num_accts']
    )

# Apply the score and then the engagement level
#df_unique['engagement_score'] = df_unique.apply(engagement_score, axis=1)

def engagement_level(score):
    if score >= 14:
        return 'High'
    elif score >= 10:
        return 'Medium'
    else:
        return 'Low'

df_unique.loc[:, 'engagement_score'] = df_unique.apply(engagement_score, axis=1)
df_unique.loc[:, 'engagement_level'] = df_unique['engagement_score'].apply(engagement_level)

#df_unique['engagement_level'] = df_unique['engagement_score'].apply(engagement_level)

control = df_unique[df_unique['Variation'] == 'Control']['engagement_score'].dropna()
test = df_unique[df_unique['Variation'] == 'Test']['engagement_score'].dropna()


In [52]:
df_unique.head

<bound method NDFrame.head of         client_id             visitor_id                      visit_id  \
385752    6187864  113539532_90731779729    77393632_6804608909_354572   
445470    8295888  238902635_81860264789    540045810_7898209765_29471   
421492    2142847  394575357_33279393282  376590287_73561231479_860065   
418424    6868690  790824480_59095058992  220987639_83305611081_845764   
423101    6506786  937099049_82984578854  603535171_56728735321_558098   
...           ...                    ...                           ...   
171591    2685910  321566510_66607009808   607150032_7201582900_558690   
205772    9584408  748244138_48778380454  484298588_81471639218_981974   
103229    6752370  258848572_66112715827  147442660_10728929690_338280   
78249     4192640  692067844_75217592829   706721307_85347845958_18583   
34734     7179755  167765295_97487764427   264484508_5982901710_928530   

       process_step            date_time Variation  clnt_tenure_yr  \
385752     

In [50]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(control, test, equal_var=False)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

T-statistic: 3.518
P-value: 0.0004


In [48]:
control.head()

385752    13.0
415184     9.0
386234    17.0
347038    18.0
404453    13.0
Name: engagement_score, dtype: float64